In [1]:
import dask_awkward as dak
import awkward as ak
from distributed import LocalCluster, Client, progress
import time
import numpy as np
import matplotlib.pyplot as plt
import json
import mplhep as hep
import glob
import pandas as pd

plt.style.use(hep.style.CMS)

client =  Client(n_workers=40,  threads_per_worker=1, processes=True, memory_limit='8 GiB') 


In [2]:
def applyRegionCatCuts(
    events,
    category: str,
    region_name: str,
    process: str,
    variation: str,
    do_vbf_filter_study: bool,
):
    use_var = "nominal" if (isinstance(variation, str) and variation.startswith("wgt")) else variation

    # Helper to fetch the right column, falling back to _nominal or base if needed
    def varcol(base):
        """
        Fetch the appropriate column from the events object, handling variations.

        Attempts to retrieve the column named '{base}_{use_var}', falling back to '{base}_nominal' and then '{base}'.
        Raises a KeyError if none of these columns are present in events.fields.

        Parameters
        ----------
        base : str
            The base name of the column to retrieve.

        Returns
        -------
        awkward.Array
            The selected column from the events object.

        Raises
        ------
        KeyError
            If none of the candidate columns are found in events.fields.
        """
        # print(f"Fetching variable column for: {base}")
        # print(f"Using variation: {use_var}")
        for cand in (f"{base}_{use_var}", f"{base}_nominal", base):
            if cand in events.fields:
                return events[cand]
        raise KeyError(
            f"[selection] Missing required field for selection: tried {base}_{use_var}, {base}_nominal, {base}"
        )

    # do mass region cut
    mass = events.dimuon_mass
    z_peak = (mass > 70) & (mass < 110)
    h_sidebands = ((mass > 110) & (mass < 115)) | ((mass > 135) & (mass < 150))
    h_peak = (mass > 115) & (mass < 135)
    if region_name == "signal":
        region = h_sidebands | h_peak
    elif region_name == "h-peak":
        region = h_peak
    elif region_name == "h-sidebands":
        region = h_sidebands
    elif region_name == "z-peak":
        region = z_peak
    else:
        print("ERROR: Invalid region specified. Acceptable regions are: signal, h-peak, h-sidebands, z-peak")
        raise ValueError

    # --- category cuts: USE varcol(...) for JES/JER-affected columns ---
    nbt_loose = varcol("nBtagLoose")
    nbt_medium = varcol("nBtagMedium")
    jj_mass = varcol("jj_mass")
    jj_dEta = varcol("jj_dEta")
    jet1_pt = varcol("jet1_pt")
    njets = varcol("njets")  # if you cut on it anywhere

    # do category cut
    if category == "nocat":
        # print("nocat mode!")
        prod_cat_cut = ak.ones_like(region, dtype="bool")
        # prod_cat_cut = ak.fill_none(events[f"jj_mass_{variation}"] > 400, value=False)
        # prod_cat_cut = prod_cat_cut & ak.fill_none(events[f"jet1_pt_{variation}"] > 35, value=False)

    else:  # VBF or ggH
        btagLoose_filter = ak.fill_none((nbt_loose >= 2), value=False)
        btagMedium_filter = ak.fill_none((nbt_medium >= 1), value=False) & ak.fill_none((njets >= 2), value=False)
        btag_cut = btagLoose_filter | btagMedium_filter
        # vbf_cut = ak.fill_none(events.vbf_cut, value=False) # in the future none values will be replaced with False
        vbf_cut = (jj_mass > 400) & (jj_dEta > 2.5) & (jet1_pt > 35)
        vbf_cut = ak.fill_none(vbf_cut, value=False)
        if category == "vbf":
            # print("vbf mode!")
            prod_cat_cut = vbf_cut
            prod_cat_cut = (
                prod_cat_cut & (~btag_cut)
            )  # btag cut is for VH and ttH categories
        elif category == "ggh":
            # print("ggH mode!")
            prod_cat_cut = ~vbf_cut
            prod_cat_cut = (
                prod_cat_cut & (~btag_cut)
            )  # btag cut is for VH and ttH categories
        else:
            print("Error: invalid category option!")
            print("Error: invalid category option! Valid options are: 'vbf', 'ggh', 'nocat'.")
            raise ValueError("Invalid category option! Valid options are: 'vbf', 'ggh', 'nocat'.")

    if do_vbf_filter_study:
        if "dy_" in process:
            vbf_filter = ak.fill_none((events.gjj_mass > 350), value=False)
            is_vbf_filter = ("dy_VBF_filter" in process) or (
                process == "dy_m105_160_vbf_amc"
            )
            if is_vbf_filter:
                # print(f"applying VBF filter cut on: {process}")

                prod_cat_cut = prod_cat_cut & vbf_filter
            else:
                # print(f"cutting off inclusive dy: {process}")
                prod_cat_cut = prod_cat_cut & ~vbf_filter
        else:
            # print(f"no extra processing for {process}")
            pass

    category_selection = prod_cat_cut & region
    # filter events fro selected category

    # print(f"len(events) {process} b4 selection: {len(events)}")
    events = events[category_selection]
    return events

In [14]:
"""
This code prints ggH/VBF channel yields after applying category cuts
"""

def applyVBF_cutV1(events):
    btag_cut =ak.fill_none((events.nBtagLoose_nominal >= 2), value=False) | ak.fill_none((events.nBtagMedium_nominal >= 1), value=False)
    vbf_cut = (events.jj_mass_nominal > 400) & (events.jj_dEta_nominal > 2.5) & (events.jet1_pt_nominal > 35) 
    vbf_cut = ak.fill_none(vbf_cut, value=False)
    dimuon_mass = events.dimuon_mass
    VBF_filter = (
        vbf_cut & 
        ~btag_cut # btag cut is for VH and ttH categories
    )
    trues = ak.ones_like(dimuon_mass, dtype="bool")
    falses = ak.zeros_like(dimuon_mass, dtype="bool")
    events["vbf_filter"] = ak.where(VBF_filter, trues,falses)
    return events[VBF_filter]


def applyGGH_NoBtagNjet1(events):
    btagLoose_filter = ak.fill_none((events.nBtagLoose_nominal >= 2), value=False)
    btagMedium_filter = ak.fill_none((events.nBtagMedium_nominal >= 1), value=False) & ak.fill_none((events.njets_nominal >= 2), value=False)
    btag_cut = (btagLoose_filter | btagMedium_filter)
    vbf_cut = (events.jj_mass_nominal > 400) & (events.jj_dEta_nominal > 2.5) & (events.jet1_pt_nominal > 35) 
    vbf_cut = ak.fill_none(vbf_cut, value=False)
    ggH_filter = (
        ~vbf_cut & 
        ~btag_cut # btag cut is for VH and ttH categories
    )
    return events[ggH_filter]

def applyGGH_30(events):
    btagLoose_filter = ak.fill_none((events.nBtagLoose_nominal >= 2), value=False)
    btagMedium_filter = ak.fill_none((events.nBtagMedium_nominal >= 1), value=False) & ak.fill_none((events.njets_nominal >= 2), value=False)
    btag_cut = (btagLoose_filter | btagMedium_filter)
    vbf_cut = (events.jj_mass_nominal > 400) & (events.jj_dEta_nominal > 2.5) & (events.jet1_pt_nominal > 35)   & (events.jet2_pt_nominal > 30) 
    vbf_cut = ak.fill_none(vbf_cut, value=False)
    jet_30_cut = ak.fill_none((events.jet1_pt_nominal > 30), value=False)
    ggH_filter = (
        ~vbf_cut 
        & ~btag_cut # btag cut is for VH and ttH categories
    )
    return events[ggH_filter]

def applyVBF_30(events):
    btag_cut =ak.fill_none((events.nBtagLoose_nominal >= 2), value=False) | ak.fill_none((events.nBtagMedium_nominal >= 1), value=False)
    vbf_cut = (events.jj_mass_nominal > 400) & (events.jj_dEta_nominal > 2.5) & (events.jet1_pt_nominal > 35)   & (events.jet2_pt_nominal > 30)
    vbf_cut = ak.fill_none(vbf_cut, value=False)
    dimuon_mass = events.dimuon_mass
    VBF_filter = (
        vbf_cut & 
        ~btag_cut # btag cut is for VH and ttH categories
    )
    trues = ak.ones_like(dimuon_mass, dtype="bool")
    falses = ak.zeros_like(dimuon_mass, dtype="bool")
    events["vbf_filter"] = ak.where(VBF_filter, trues,falses)
    return events[VBF_filter]


def applyGGH_cutflow(events):
    btagLoose_filter = ak.fill_none((events.nBtagLoose_nominal >= 2), value=False)
    btagMedium_filter = ak.fill_none((events.nBtagMedium_nominal >= 1), value=False) & ak.fill_none((events.njets_nominal >= 2), value=False)
    btag_cut = btagLoose_filter | btagMedium_filter
    vbf_cut = (events.jj_mass_nominal > 400) & (events.jj_dEta_nominal > 2.5) & (events.jet1_pt_nominal > 35) 
    vbf_cut = ak.fill_none(vbf_cut, value=False)
    dimuon_mass = events.dimuon_mass
    ggH_filter = (
        ~vbf_cut & 
        ~btag_cut # btag cut is for VH and ttH categories
    )
    return events[ggH_filter]

def applyGGH_noJetPt(events):
    btag_cut =ak.fill_none((events.nBtagLoose_nominal >= 2), value=False) | ak.fill_none((events.nBtagMedium_nominal >= 1), value=False)
    vbf_cut = (events.jj_mass_nominal > 400) & (events.jj_dEta_nominal > 2.5)
    vbf_cut = ak.fill_none(vbf_cut, value=False)
    dimuon_mass = events.dimuon_mass
    ggH_filter = (
        ~vbf_cut & 
        ~btag_cut # btag cut is for VH and ttH categories
    )
    return events[ggH_filter]

def veto_ttH_VH(events):
    btagLoose_filter = ak.fill_none((events.nBtagLoose_nominal >= 2), value=False)
    btagMedium_filter = ak.fill_none((events.nBtagMedium_nominal >= 1), value=False) & ak.fill_none((events.njets_nominal >= 2), value=False)
    btag_cut = btagLoose_filter | btagMedium_filter
    
    bool_filter = (
        ~btag_cut # btag cut is for VH and ttH categories
    )
    return events[bool_filter]


def veto_nJetGeq3(events):
    njet_filter = ak.fill_none((events.njets_nominal <= 2), value=True)
    bool_filter = (
        njet_filter # btag cut is for VH and ttH categories
    )
    return events[bool_filter]

def filterRegion(events, region="h-peak"):
    dimuon_mass = events.dimuon_mass
    if region =="h-peak":
        region = (dimuon_mass > 115) & (dimuon_mass < 135)
    elif region =="h-sidebands":
        region = ((dimuon_mass > 110) & (dimuon_mass < 115)) | ((dimuon_mass > 135) & (dimuon_mass < 150))
    elif region =="signal":
        region = (dimuon_mass >= 110) & (dimuon_mass <= 150.0)
    elif region =="z-peak":
        region = (dimuon_mass >= 70) & (dimuon_mass <= 110.0)
    elif region =="combined":
        region = (dimuon_mass >= 70) & (dimuon_mass <= 150.0)

    # mu1_pt = events.mu1_pt
    # mu1ptOfInterest = (mu1_pt > 75) & (mu1_pt < 150.0)
    # events = events[region&mu1ptOfInterest]
    events = events[region]
    return events

V1_fields_2compute = [
    "wgt_nominal",
    "nBtagLoose_nominal",
    "nBtagMedium_nominal",
    "mu1_pt",
    "mu2_pt",
    "mu1_eta",
    "mu2_eta",
    "mu1_phi",
    "mu2_phi",
    "dimuon_pt",
    "dimuon_eta",
    "dimuon_phi",
    "dimuon_mass",
    "jet1_phi_nominal",
    "jet1_pt_nominal",
    "jet2_pt_nominal",
    "jet2_phi_nominal",
    "jet1_eta_nominal",
    "jet2_eta_nominal",
    "jj_mass_nominal",
    "jj_dEta_nominal",
    # "region",
    "event",
    "njets_nominal",
    # "run",
    # "event",
    # "luminosityBlock",
    "gjj_mass",
]
 
#

In [21]:
# year = "2018"
year = "2018PR"
# year="*"
# year = "2017"
# year = "2016*"
# label="V2_Jan29_JecOn_TrigMatchFixed_2016UlJetIdFix"

# label="DYamcNLO_11Apr2025"
# label="UpdatedDY_100_200_CrossSection_24Feb_jetpuidOff"
# label="test_test"
# label="DYMiNNLO_30Mar2025"
# label="DYMiNNLO_11Apr2025"
# label="DYMiNNLO_HemVetoOff_17Apr2025"
# label="DYMiNNLO_HemVetoOff_18Apr2025_singleMuTrigMatch"
# label="jetHornStudy_29Apr2025_JecOnJerOff"
# # label="jetHornStudy_29Apr2025_JecOnJerStrat2_jetHornPtCut50"
# label="jetHornStudy_29Apr2025_JecOnJerStrat1n2_jetHornTightPuId"
# label="fullRun_May30_2025"
# label="fullRun_Jun21_2025"
# label="fullRun_Jun25_2025_DefJESJER"
# label="fullRun_Jun23_2025_1n2Revised"
# label="fullRun_Jul17_2025_qglFixed"
# label="JetPtCutDiffSept22_2025"
# label="Sept22_2025"
# label="JetPtCutDiffSept22_2025"
# label="synchOct10_2025"
label="synchOct14_2025"

# # year = "2022preEE"
# # label="Run3_nanoAODv12_TEST"
load_path =f"/depot/cms/users/yun79/hmm/copperheadV1clean/{label}/stage1_output/{year}/f1_0"
# load_path =f"/depot/cms/users/yun79/hmm/copperheadV1clean/{label}/stage1_output/{year}/f0_2"
# load_path =f"/depot/cms/users/yun79/hmm/copperheadV1clean/{label}/stage1_output/{year}/*"



# label="May28_NanoV12"
# load_path =f"/depot/cms/users/shar1172/hmm/copperheadV1clean/{label}/stage1_output/{year}/*"

# # events_data = dak.from_parquet(f"{load_path}/data_D/*.parquet")
# # events_data = dak.from_parquet(f"{load_path}/data_F/*.parquet")
# filelist = glob.glob(f"{load_path}/data_*")
# filelist = glob.glob(f"{load_path}/ggh_powheg*")
# filelist = glob.glob(f"{load_path}/vbf_powheg_dipole")
# filelist = glob.glob(f"{load_path}/*powheg*")
# print(filelist)
filelist = glob.glob(f"{load_path}/dy*")
# filelist = glob.glob(f"{load_path}/ttjets*")
# filelist = glob.glob(f"{load_path}/*top*")
# filelist = glob.glob(f"{load_path}/*ewk*")
# filelist = glob.glob(f"{load_path}/dy_M-100To200_MiNNLO")

total_integral = 0
for file in filelist:
    print(f"file: {file}")
    events_data = dak.from_parquet(f"{file}/*/*.parquet")
    events_data = ak.zip({field: events_data[field] for field in V1_fields_2compute}).compute()
    events_data = filterRegion(events_data, region="h-sidebands")
    # events_data = applyGGH_NoBtagNjet1(events_data)
    # events_data = applyVBF_cutV1(events_data)
    # events_data = veto_ttH_VH(events_data)
    
    events_data = applyRegionCatCuts(events_data, "ggh", "h-sidebands", "dy_", "nominal", True)


    
    # wgts = ak.fill_none(events_data.wgt_nominal, value=1.0)
    # wgts = ak.ones_like(wgts)
    wgts = ak.fill_none(events_data.wgt_nominal, value=0.0)
    data_yield = ak.sum(wgts)
    df = pd.DataFrame({field: ak.fill_none(events_data[field], value=-999.9) for field in events_data.fields})
    print(f"data_yield for {file}: {data_yield}")
    total_integral += data_yield
total_integral


file: /depot/cms/users/yun79/hmm/copperheadV1clean/synchOct14_2025/stage1_output/2018PR/f1_0/dy_M-100To200_aMCatNLO
data_yield for /depot/cms/users/yun79/hmm/copperheadV1clean/synchOct14_2025/stage1_output/2018PR/f1_0/dy_M-100To200_aMCatNLO: 292148.45529218967


np.float64(292148.45529218967)

In [ ]:



# # events_data = dak.from_parquet(f"{load_path}/data_D/*.parquet")
# # events_data = dak.from_parquet(f"{load_path}/data_F/*.parquet")
# # filelist = glob.glob(f"{load_path}/data_F")
# # filelist = glob.glob(f"{load_path}/data_*")
# filelist = glob.glob(f"{load_path}/data_*")
# filelist = glob.glob(f"{load_path}/vbf_powheg_dipole")
# filelist = glob.glob(f"{load_path}/data_D")
# print(filelist)
# filelist = glob.glob(f"{load_path}/dy*")
filelist = glob.glob(f"{load_path}/dy*100*")

total_integral = 0
for file in filelist:
    print(f"file: {file}")
    events_data = dak.from_parquet(f"{file}/*/*.parquet")
    events_data = ak.zip({field: events_data[field] for field in V1_fields_2compute}).compute()
    events_data = filterRegion(events_data, region="signal")
    # events_data = applyGGH_cutV1(events_data)
    # events_data = applyGGH_NoBtagNjet1(events_data)
    # events_data = veto_ttH_VH(events_data)
    events_data = applyVBF_cutV1(events_data)
    
    # events_data = applyGGH_30(events_data)
    # events_data = applyVBF_30(events_data)
    



    
    # data_yield = ak.sum(events_data.wgt_nominal, axis=0)
    wgts = ak.fill_none(events_data.wgt_nominal, value=1.0)
    data_yield = ak.sum(wgts)
    df = pd.DataFrame({field: ak.fill_none(events_data[field], value=-999.9) for field in events_data.fields})
    print(f"data_yield for {file}: {data_yield}")
    total_integral += data_yield
total_integral


In [10]:
label="fullRun_Jun23_2025_1n2Revised"
year="2018"
load_path =f"/depot/cms/users/yun79/hmm/copperheadV1clean/{label}/stage1_output/{year}/f1_0"
filelist = glob.glob(f"{load_path}/dy*100*")

total_integral = 0
file = filelist[0]
print(f"file: {file}")
events_data = dak.from_parquet(f"{file}/*/*.parquet")
fields = events_data.fields
wgt_fields = []
for field in fields:
    if "separate" in field:
        wgt_fields.append(field)

wgt_fields

file: /depot/cms/users/yun79/hmm/copperheadV1clean/fullRun_Jun23_2025_1n2Revised/stage1_output/2018/f1_0/dy_M-100To200_MiNNLO


['separate_wgt_genWeight',
 'separate_wgt_genWeight_normalization',
 'separate_wgt_xsec',
 'separate_wgt_lumi',
 'separate_wgt_pu_wgt',
 'separate_wgt_muID',
 'separate_wgt_muIso',
 'separate_wgt_muTrig',
 'separate_wgt_LHERen',
 'separate_wgt_LHEFac',
 'separate_wgt_pdf_2rms',
 'separate_wgt_jetpuid_wgt',
 'separate_wgt_qgl_wgt',
 'separate_wgt_zpt_wgt']

In [11]:
filelist

['/depot/cms/users/yun79/hmm/copperheadV1clean/fullRun_Jun23_2025_1n2Revised/stage1_output/2018/f1_0/dy_M-100To200_MiNNLO']

In [ ]:
ak.max(events_data.njets_nominal)

In [ ]:
events_data.jet1_pt_nominal
# events_data.njets_nominal

In [ ]:
print(events_data.njets_nominal[:50] <=2)
print(ak.fill_none(events_data.njets_nominal[:50] <=2, value=True))

In [ ]:
year = "2018"
# label="V2_Jan29_JecOn_TrigMatchFixed_2016UlJetIdFix"
# label="DYMiNNLO_30Mar2025"
label="jetHornStudy_29Apr2025_JecOnJerOff"

# label="test_test"
# year = "2022preEE"
# label="Run3_nanoAODv12_TEST"
load_path =f"/depot/cms/users/yun79/hmm/copperheadV1clean/{label}/stage1_output/{year}/f1_0"

# filelist = glob.glob(f"{load_path}/dy*")
filelist = glob.glob(f"{load_path}/dy_M-50_MiNNLO")

total_integral = 0
for file in filelist:
    print(f"file: {file}")
    events_data = dak.from_parquet(f"{file}/*/*.parquet")
    # events_data = filterRegion(events_data, region="signal")
    events_data = filterRegion(events_data, region="z-peak")
    wgt = events_data.wgt_nominal.compute()
    # print(f"wgt sum: {wgt}")
    print(f"wgt sum: {ak.sum(wgt)}")
    comp = ak.ones_like(wgt)
    for field in events_data.fields:
        if "separate" in field:
            value = events_data[field].compute()
            print(f"{field} arr: {value}")
            comp = comp*value
            # print(f"{field} curent wgt: {comp}")
    # diff = comp- wgt
    # print(f"comp : {comp}")
    # print(f"wgt : {wgt}")
    # print(f"sum wgt : {ak.sum(wgt)}")
    # print(f"difference : {diff}")
            # print(f"{field} max val: {ak.max(value)}")

In [ ]:
2.36e+03 * 228348879
41,158,111.73464724
191,709,872

In [ ]:
2.5292969635125805e+20 

In [ ]:
wgt_nominal = events_data["wgt_nominal"].compute()
ak.sum(wgt_nominal)

In [ ]:
test = wgt_nominal/ events_data["separate_wgt_qgl_wgt"].compute()
ak.sum(test)

In [ ]:
gen_wgt = events_data["separate_wgt_genWeight"].compute()
ak.sum(gen_wgt)

In [ ]:
ak.sum(gen_wgt)*7.1e-12

In [ ]:
events_data["separate_wgt_genWeight_normalization"].compute()

In [ ]:
ak.sum(events_data["wgt_nominal"].compute())

In [13]:
np.diff([0.0, 0.27, 0.64, 0.83, 0.94, 1.0])

array([0.27, 0.37, 0.19, 0.11, 0.06])

In [14]:
np.array([0.27, 0.37, 0.19, 0.11, 0.06]).sum()

1.0

In [31]:
import dask_awkward as dak
import awkward as ak

events = dak.from_parquet("/depot/cms/users/yun79/hmm/copperheadV1clean/fullRun_Jun23_2025_1n2Revised/V2_Aug28_PosWgtRun0p7_MassResRun1_ggh/stage2_output/2016*/processed_events_data.parquet")

In [32]:
events.compute()

<Array [{dimuon_mass: 90.6, ...}, ..., {...}] type='25550270 * {dimuon_mass...'>

In [27]:
len(events.fields)

6

In [22]:
events.fields

['wgt_nominal',
 'dimuon_mass',
 'subCategory_idx',
 'event',
 'BDT_score',
 'dimuon_ebe_mass_res',
 'wgt_l1prefiring_up',
 'wgt_l1prefiring_down']

In [17]:
events = dak.from_parquet("/depot/cms/users/yun79/hmm/copperheadV1clean/fullRun_Jun23_2025_1n2Revised/V2_Aug28_PosWgtRun0p7_MassResRun1_ggh/stage2_output/2017/processed_events_data.parquet")
len(events.fields)

6

In [18]:
events.fields

['dimuon_mass',
 'wgt_nominal',
 'dimuon_ebe_mass_res',
 'subCategory_idx',
 'event',
 'BDT_score']

In [19]:
events.fields

['dimuon_mass',
 'wgt_nominal',
 'dimuon_ebe_mass_res',
 'subCategory_idx',
 'event',
 'BDT_score']

In [33]:
import dask_awkward as dak


load_path = "/depot/cms/users/yun79/hmm/copperheadV1clean/fullRun_Jun23_2025_1n2Revised/V2_Aug28_PosWgtRun0p7_MassResRun1_ggh/stage2_output/2018/processed_events_sigMC_ggh.parquet"
events = dak.from_parquet(load_path)
events.fields

['wgt_nominal',
 'dimuon_mass',
 'subCategory_idx',
 'event',
 'BDT_score',
 'dimuon_ebe_mass_res',
 'wgt_l1prefiring_up',
 'wgt_l1prefiring_down']

In [34]:
fields2load = ['nBtagLoose_BBEC1_2018_up', 'jet2_pt_BBEC1_2018_up', 'jj_mass_BBEC1_up', 'nBtagMedium_Absolute_2018_up', 'mmj2_dPhi_Absolute_2018_up', 'mmj_min_dPhi_BBEC1_up', 'mmj_min_dEta_BBEC1_2018_up', 'jj_dEta_BBEC1_up', 'njets_Absolute_up', 'mmj1_dPhi_nominal', 'jj_mass_Absolute_up', 'rpt_Absolute_2018_up', 'nBtagMedium_nominal', 'jet1_eta_Absolute_up', 'jj_dPhi_nominal', 'nBtagLoose_Absolute_2018_up', 'zeppenfeld_BBEC1_up', 'jet1_eta_BBEC1_2018_up', 'jj_dEta_nominal', 'jet2_eta_Absolute_2018_up', 'mu1_pt_over_mass', 'mmj2_dPhi_nominal', 'mmj_min_dPhi_Absolute_2018_up', 'mmj_min_dEta_Absolute_up', 'dimuon_pt', 'njets_nominal', 'jet1_eta_Absolute_2018_up', 'jj_dPhi_BBEC1_2018_up', 'jet1_eta_nominal', 'njets_BBEC1_up', 'year', 'dimuon_rapidity', 'mmj_min_dEta_Absolute_2018_up', 'mmj_min_dEta_BBEC1_up', 'jj_dEta_BBEC1_2018_up', 'rpt_BBEC1_2018_up', 'njets_Absolute_2018_up', 'nBtagMedium_Absolute_up', 'jj_mass_nominal', 'njets_BBEC1_2018_up', 'jet2_eta_Absolute_up', 'nBtagMedium_BBEC1_up', 'mmj_min_dPhi_Absolute_up', 'mu1_eta', 'mmj2_dEta_Absolute_up', 'mmj1_dEta_BBEC1_2018_up', 'jj_dEta_Absolute_up', 'jet2_eta_BBEC1_2018_up', 'wgt_nominal', 'nBtagMedium_BBEC1_2018_up', 'jet1_pt_nominal', 'jet1_pt_Absolute_up', 'dimuon_phi_cs', 'jet2_eta_nominal', 'zeppenfeld_Absolute_2018_up', 'dimuon_ebe_mass_res', 'rpt_nominal', 'mu2_eta', 'nBtagLoose_Absolute_up', 'jj_dPhi_Absolute_up', 'mmj1_dPhi_BBEC1_up', 'rpt_Absolute_up', 'mmj1_dEta_nominal', 'jet1_pt_BBEC1_up', 'mmj1_dPhi_BBEC1_2018_up', 'mmj1_dPhi_Absolute_up', 'mmj2_dEta_Absolute_2018_up', 'mmj2_dEta_nominal', 'event', 'zeppenfeld_Absolute_up', 'zeppenfeld_nominal', 'mmj_min_dPhi_BBEC1_2018_up', 'jj_mass_Absolute_2018_up', 'mmj2_dPhi_Absolute_up', 'jj_dPhi_BBEC1_up', 'jj_dEta_Absolute_2018_up', 'jet1_eta_BBEC1_up', 'jj_mass_BBEC1_2018_up', 'mmj2_dPhi_BBEC1_up', 'mmj_min_dPhi_nominal', 'rpt_BBEC1_up', 'mmj1_dEta_Absolute_2018_up', 'jet2_pt_nominal', 'mmj2_dEta_BBEC1_up', 'mmj1_dEta_BBEC1_up', 'jet2_pt_BBEC1_up', 'mmj1_dPhi_Absolute_2018_up', 'mmj1_dEta_Absolute_up', 'zeppenfeld_BBEC1_2018_up', 'dimuon_mass', 'nBtagLoose_BBEC1_up', 'dimuon_cos_theta_cs', 'jet1_pt_BBEC1_2018_up', 'jet2_eta_BBEC1_up', 'jet2_pt_Absolute_2018_up', 'mmj_min_dEta_nominal', 'mmj2_dPhi_BBEC1_2018_up', 'jet1_pt_Absolute_2018_up', 'jj_dPhi_Absolute_2018_up', 'jet2_pt_Absolute_up', 'nBtagLoose_nominal', 'mu2_pt_over_mass', 'mmj2_dEta_BBEC1_2018_up']
field = "nBtagLoose_BBEC1_2018_up"
field = field.split("_")[0]
field

'nBtagLoose'

In [40]:
events = dak.from_parquet("/depot/cms/users/yun79/hmm/copperheadV1clean/Sept22_2025/stage1_output/2018/f0_2/dy_M-50_MiNNLO/*/*.parquet")

In [41]:
events.fields

['event',
 'PV_npvs',
 'PV_npvsGood',
 'MET_pt',
 'MET_phi',
 'MET_sumEt',
 'mu1_pt',
 'mu1_ptErr',
 'mu2_pt',
 'mu2_ptErr',
 'mu1_pt_over_mass',
 'mu2_pt_over_mass',
 'mu1_eta',
 'mu2_eta',
 'mu1_phi',
 'mu2_phi',
 'mu1_charge',
 'mu2_charge',
 'mu1_iso',
 'mu2_iso',
 'nmuons',
 'dimuon_mass',
 'dimuon_pt',
 'dimuon_pt_log',
 'dimuon_eta',
 'dimuon_rapidity',
 'dimuon_phi',
 'dimuon_dEta',
 'dimuon_dPhi',
 'dimuon_dR',
 'acoplanarity',
 'dimuon_ebe_mass_res',
 'dimuon_ebe_mass_res_rel',
 'uncalibrated_dimuon_ebe_mass_res',
 'dimuon_cos_theta_cs',
 'dimuon_phi_cs',
 'dimuon_cos_theta_eta',
 'dimuon_phi_eta',
 'mu1_pt_raw',
 'mu2_pt_raw',
 'mu1_pt_fsr',
 'mu2_pt_fsr',
 'year',
 'run',
 'luminosityBlock',
 'fraction',
 'jet1_default_pt_nominal',
 'jet1_default_eta_nominal',
 'jet2_default_pt_nominal',
 'jet2_default_eta_nominal',
 'gjet1_pt',
 'gjet1_eta',
 'gjet1_phi',
 'gjet1_mass',
 'gjet2_pt',
 'gjet2_eta',
 'gjet2_phi',
 'gjet2_mass',
 'gjj_pt',
 'gjj_eta',
 'gjj_phi',
 'gjj_mass',
